# Ladle Detection — Model Evaluation

**Dataset:** `Ladle_detection_test_v1i` — 415 images, YOLOv8-OBB format  
**Model:** Custom-trained `best.pt` (YOLOv8-OBB, trained December 2025 for SIH)  
**Task:** Oriented bounding box detection of steel ladles in industrial environments

This notebook:
1. Runs `model.val()` on the held-out **test split** (18 images)
2. Visualises OBB predictions on sample images
3. Plots a precision-recall curve
4. Demonstrates the colour-strip decoding pipeline on a sample frame

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
from ultralytics import YOLO

# Make src importable from the notebook
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from color_identification import detect_strips, strips_to_ladle_id

# Paths
REPO_ROOT   = Path.cwd().parent
MODEL_PATH  = REPO_ROOT / 'best.pt'
DATA_YAML   = REPO_ROOT / 'dataset' / 'data.yaml'   # unzip the dataset here first
TEST_IMAGES = REPO_ROOT / 'dataset' / 'test' / 'images'

print(f'Model  : {MODEL_PATH}  (exists={MODEL_PATH.exists()})')
print(f'Dataset: {DATA_YAML}   (exists={DATA_YAML.exists()})')

## 1  Load Model

In [ ]:
model = YOLO(str(MODEL_PATH))
print('Model task     :', model.task)
print('Class names    :', model.names)
print('Model params   :', f"{sum(p.numel() for p in model.model.parameters()):,}")

## 2  Quantitative Evaluation on Test Split

> **Note:** unzip `Ladle_detection_test_v1i_yolov8-obb.zip` into `dataset/` first.  
> `unzip Ladle_detection_test_v1i_yolov8-obb.zip -d dataset/`

In [ ]:
metrics = model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=512,
    conf=0.5,
    verbose=True,
)

print('\n── Key Metrics ─────────────────────────────────────')
print(f'mAP@50      : {metrics.box.map50:.4f}')
print(f'mAP@50-95   : {metrics.box.map:.4f}')
print(f'Precision   : {metrics.box.mp:.4f}')
print(f'Recall      : {metrics.box.mr:.4f}')

## 3  Visualise Predictions on Sample Test Images

In [ ]:
sample_images = sorted(TEST_IMAGES.glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_images):
    results = model.predict(str(img_path), conf=0.5, verbose=False)
    annotated = results[0].plot()           # BGR
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.set_title(img_path.stem[:30], fontsize=7)
    ax.axis('off')

plt.suptitle('YOLOv8-OBB Ladle Detection — Test Set Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'assets' / 'sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → assets/sample_predictions.png')

## 4  Confidence Distribution Across Test Set

In [ ]:
all_confs = []
for img_path in sorted(TEST_IMAGES.glob('*.jpg')):
    results = model.predict(str(img_path), conf=0.1, verbose=False)
    for r in results:
        if r.boxes is not None:
            all_confs.extend(r.boxes.conf.tolist())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_confs, bins=20, color='#2563eb', edgecolor='white', alpha=0.85)
ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='Threshold = 0.50')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Count')
ax.set_title('Detection Confidence Distribution — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig(REPO_ROOT / 'assets' / 'confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total detections (conf > 0.1): {len(all_confs)}')
print(f'Median confidence           : {np.median(all_confs):.3f}')

## 5  Colour-Strip Decoding Demo

Demonstrates the full ID-decoding pipeline: detect ladle → crop ROI → run HSV strip detection → read ladle number.

In [ ]:
demo_img_path = sorted(TEST_IMAGES.glob('*.jpg'))[0]
frame = cv2.imread(str(demo_img_path))

results = model.predict(frame, conf=0.5, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: full frame with OBB annotation
annotated = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)
axes[0].imshow(annotated)
axes[0].set_title('YOLOv8-OBB Detection', fontweight='bold')
axes[0].axis('off')

# Right: cropped ROI + strip decode result
decoded_id = None
for r in results:
    if r.boxes is None:
        continue
    for box in r.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        h, w = frame.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w-1, x2), min(h-1, y2)
        if x2 > x1 and y2 > y1:
            roi = frame[y1:y2, x1:x2]
            strips = detect_strips(roi, x1, y1)
            decoded_id, digits = strips_to_ladle_id(strips)
            axes[1].imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
            axes[1].set_title(
                f'Ladle ROI — Strips: {digits} → ID: {decoded_id}',
                fontweight='bold'
            )
            axes[1].axis('off')
            break
    break

plt.suptitle('End-to-End: Detection + Colour-Strip Decoding', fontsize=12)
plt.tight_layout()
plt.show()
print(f'Decoded Ladle ID: {decoded_id}')

## 6  Why OBB?

Standard YOLO uses axis-aligned boxes (AABB).  
Ladles in a steel plant can be:
- Tilted during pouring
- At angles on conveyor rails
- Partially occluded by crane equipment

Oriented Bounding Boxes (OBB) tightly wrap rotated objects, reducing the box area that bleeds into background pixels. This matters for colour-strip detection: a looser AABB would include adjacent ladles or background refractory material, polluting the HSV histogram and causing false-positive strip detections.

The dataset was annotated in YOLOv8-OBB format (8-point polygon coordinates normalised to image dimensions).